# Qwen3-8B-Base + SAE PTSD vs Control 特征分布对比2

目标：对 Thousand Voices of Trauma、EmpatheticDialogues、DailyDialog 使用同一层 Qwen3 residual 和同一个 SAE 做特征提取，然后比较 PTSD 对话相对于两个对照集的 feature 分布。

为了避免 PTSD conversation 比 control conversation 长很多造成不公平，本 notebook 会先把三组文本按 token 总量均衡；模型计算时只用固定长度 windows 防止输入过长，统计时以 token 为单位汇总。


In [1]:
# 如果远程环境缺包，取消注释安装。Qwen3 需要 transformers >= 4.51.0。
# %pip install -U 'transformers>=4.51.0' accelerate safetensors pandas tqdm

import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)
print('torch:', torch.__version__)


/inspire/hdd/project/fdu-aidake-cfff/public/.conda/envs/GP35/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0+cu128


## 1. 配置

`DATA_DIR = PROJECT_DIR / 'thousand-voices-trauma'` 保持固定。对照数据直接读取你手动下载并解压好的 DailyDialog 和 EmpatheticDialogues 原始文件。


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'thousand-voices-trauma').exists() and (PROJECT_DIR.parent / 'thousand-voices-trauma').exists():
    PROJECT_DIR = PROJECT_DIR.parent

MODEL_DIR = PROJECT_DIR / 'qwen3-8b-base'
SAE_DIR = PROJECT_DIR / 'qwen3-8b-SAE'
DATA_DIR = PROJECT_DIR / 'thousand-voices-trauma'
DAILY_TEXT_PATH = PROJECT_DIR / 'daily_dialogues' / 'dialogues_text.txt'
EMPATHETIC_DIR = PROJECT_DIR / 'empathetic_dialogues'

CONVERSATION_DIR = DATA_DIR / 'conversations'
METADATA_DIR = DATA_DIR / 'metadata'
TEXT_COLUMN = 'text'

# 批量测试入口：填你关心的 SAE 层，例如 [12, 16, 20, 24, 28]。
TARGET_LAYERS = [12,16,20,24,28,32]
TARGET_LAYER = TARGET_LAYERS[0]
SAE_TOP_K = 50
TOP_N = 50
MAX_LENGTH = 512
WINDOW_TOKEN_LENGTH = 512
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# 三组数据先按完整 conversation tokenize，再按 token 总量采样；WINDOW_TOKEN_LENGTH 只控制模型计算窗口，不参与统计定义。
# None 表示按三组中最小 token 总量自动均衡；也可以手动设成整数，例如 2000。
# 这里的数值是每个 dataset 的 token 上限；如果想三组总共约 2000，可设成 2000 // 3。
BALANCED_TOKENS_PER_DATASET = 40000
RANDOM_SEED = 42

OUTPUT_BASE_DIR = PROJECT_DIR / 'outputs'

print('PROJECT_DIR:', PROJECT_DIR)
print('MODEL_DIR exists:', MODEL_DIR.exists())
print('SAE_DIR exists:', SAE_DIR.exists())
print('DATA_DIR exists:', DATA_DIR.exists())
print('DAILY_TEXT_PATH exists:', DAILY_TEXT_PATH.exists())
print('EMPATHETIC_DIR exists:', EMPATHETIC_DIR.exists())
print('TARGET_LAYERS:', TARGET_LAYERS)
print('OUTPUT_BASE_DIR:', OUTPUT_BASE_DIR)


PROJECT_DIR: /inspire/hdd/project/fdu-aidake-cfff/public/Group35
MODEL_DIR exists: True
SAE_DIR exists: True
DATA_DIR exists: True
DAILY_TEXT_PATH exists: True
EMPATHETIC_DIR exists: True
OUTPUT_DIR: /inspire/hdd/project/fdu-aidake-cfff/public/Group35/outputs/ptsd_control_sae_layer20


## 2. 读取并切分 PTSD、DailyDialog、EmpatheticDialogues

DailyDialog 用 `dialogues_text.txt`，每行是一条 conversation，`__eou__` 分隔轮次。EmpatheticDialogues 用 `train.csv/valid.csv/test.csv`，按 `conv_id` 聚合 utterance，并清洗 `_comma_` 标记。随后三组文本都会按 tokenizer 计算完整 token 数，比较时按 token 总量均衡。


In [ ]:
def parse_conversation_name(path: Path) -> tuple[str, str | None, str | None]:
    conversation_id = path.stem
    for suffix in ('_conversation', '_conversations'):
        if conversation_id.endswith(suffix):
            conversation_id = conversation_id[:-len(suffix)]
            break

    parts = conversation_id.split('_', 1)
    subject_id = parts[0] if len(parts) == 2 else None
    phase = parts[1] if len(parts) == 2 else None
    return conversation_id, subject_id, phase


def join_turns(turns) -> str:
    return chr(10).join(str(turn).strip() for turn in turns if str(turn).strip())


def clean_control_text(text) -> str:
    text = '' if text is None else str(text)
    text = text.replace('_comma_', ',').replace('_comma', ',')
    return ' '.join(text.split())


def conversation_text_from_payload(payload: dict) -> str:
    turns = payload.get('full_conversation') or payload.get('conversation')
    if turns is None and payload.get('three_turn_sequences'):
        seen = []
        seen_set = set()
        for seq in payload['three_turn_sequences']:
            for turn in seq:
                if turn not in seen_set:
                    seen.append(turn)
                    seen_set.add(turn)
        turns = seen

    if isinstance(turns, list):
        return join_turns(turns)
    if isinstance(turns, str):
        return turns.strip()
    raise ValueError('Conversation JSON must contain full_conversation, conversation, or three_turn_sequences.')


def metadata_for_ptsd_conversation(conversation_id: str) -> dict:
    metadata_path = METADATA_DIR / f'{conversation_id}_metadata.json'
    if not metadata_path.exists():
        return {}

    with metadata_path.open('r', encoding='utf-8') as f:
        metadata = json.load(f)

    client = metadata.get('client_profile', {})
    trauma = metadata.get('trauma_info', {})
    return {
        'client_age_group': client.get('age_group'),
        'client_gender': client.get('gender'),
        'co_occurring_condition': client.get('co_occurring_condition'),
        'exhibited_behaviors': ';'.join(client.get('exhibited_behaviors', [])),
        'trauma_type': trauma.get('type'),
        'session_topic': trauma.get('session_topic'),
    }


def load_ptsd_conversations() -> pd.DataFrame:
    if not CONVERSATION_DIR.exists():
        raise FileNotFoundError(f'Conversation directory does not exist: {CONVERSATION_DIR}')

    paths = sorted(p for p in CONVERSATION_DIR.glob('*.json') if p.is_file())
    if not paths:
        raise FileNotFoundError(f'No conversation JSON files found in: {CONVERSATION_DIR}')

    rows = []
    for path in tqdm(paths, desc='load PTSD conversations'):
        with path.open('r', encoding='utf-8') as f:
            payload = json.load(f)

        text = conversation_text_from_payload(payload)
        if not text:
            continue

        conversation_id, subject_id, phase = parse_conversation_name(path)
        rows.append({
            'dataset': 'ptsd',
            'sample_id': f'ptsd:{conversation_id}',
            'conversation_id': conversation_id,
            'subject_id': subject_id,
            'phase': phase,
            'source_split': None,
            'source_file': str(path.relative_to(DATA_DIR)),
            TEXT_COLUMN: text,
            **metadata_for_ptsd_conversation(conversation_id),
        })
    return pd.DataFrame(rows)


def load_daily_dialog_conversations() -> pd.DataFrame:
    if not DAILY_TEXT_PATH.exists():
        raise FileNotFoundError(f'Missing DailyDialog text file: {DAILY_TEXT_PATH}')

    rows = []
    with DAILY_TEXT_PATH.open('r', encoding='utf-8', errors='replace') as f:
        for idx, line in enumerate(f):
            turns = [clean_control_text(turn) for turn in line.split('__eou__')]
            text = join_turns(turns)
            if not text:
                continue
            rows.append({
                'dataset': 'daily',
                'sample_id': f'daily:{idx}',
                'conversation_id': str(idx),
                'subject_id': None,
                'phase': None,
                'source_split': 'all',
                'source_file': str(DAILY_TEXT_PATH.relative_to(PROJECT_DIR)),
                TEXT_COLUMN: text,
            })
    return pd.DataFrame(rows)


def iter_empathetic_rows(path: Path):
    with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            # Some valid/test rows contain extra recommendation/tag columns. The first 8 fields are the real schema.
            if len(row) < 8:
                continue
            row = row[:8]
            yield dict(zip(header[:8], row))


def load_empathetic_conversations() -> pd.DataFrame:
    paths = [EMPATHETIC_DIR / 'train.csv', EMPATHETIC_DIR / 'valid.csv', EMPATHETIC_DIR / 'test.csv']
    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Missing EmpatheticDialogues csv files: {missing}')

    rows = []
    for path in paths:
        source_split = 'validation' if path.stem == 'valid' else path.stem
        grouped = defaultdict(list)
        for item in iter_empathetic_rows(path):
            grouped[str(item['conv_id'])].append(item)

        for conv_id, items in tqdm(grouped.items(), desc=f'load Empathetic {source_split}'):
            items = sorted(items, key=lambda x: int(x['utterance_idx']))
            turns = [clean_control_text(item['utterance']) for item in items]
            text = join_turns(turns)
            if not text:
                continue
            first = items[0]
            rows.append({
                'dataset': 'empathetic',
                'sample_id': f'empathetic:{source_split}:{conv_id}',
                'conversation_id': conv_id,
                'subject_id': None,
                'phase': None,
                'source_split': source_split,
                'source_file': str(path.relative_to(PROJECT_DIR)),
                'context': clean_control_text(first.get('context')),
                'prompt': clean_control_text(first.get('prompt')),
                TEXT_COLUMN: text,
            })
    return pd.DataFrame(rows)


ptsd_conversation_df = load_ptsd_conversations()
daily_conversation_df = load_daily_dialog_conversations()
empathetic_conversation_df = load_empathetic_conversations()

conversation_df = pd.concat(
    [ptsd_conversation_df, daily_conversation_df, empathetic_conversation_df],
    ignore_index=True,
    sort=False,
)
print('raw conversations:')
print(conversation_df.groupby('dataset').size())
print('total conversations:', len(conversation_df))


load Empathetic test: 100%|██████████| 2540/2540 [00:00<00:00, 23070.63it/s]


raw conversations:
dataset
daily         13118
empathetic    23143
ptsd           3000
dtype: int64
total conversations: 39261


## 3. 加载模型、tokenizer 和 SAE


In [3]:
def require_path(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f'{name} does not exist: {path}')


require_path(MODEL_DIR, 'MODEL_DIR')
require_path(SAE_DIR, 'SAE_DIR')
SAE_SNAPSHOT_DIR = SAE_DIR / 'snapshots' / '9b7d647127c303dd371e5dbd992236bd9799b171'
require_path(SAE_SNAPSHOT_DIR, 'SAE snapshot directory')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=DTYPE,
    device_map='auto',
    trust_remote_code=True,
    local_files_only=True,
)
model.eval()

print('model dtype:', DTYPE)

def load_sae_for_layer(layer: int):
    global TARGET_LAYER, layer_device, W_enc, b_enc
    TARGET_LAYER = int(layer)
    sae_file = SAE_SNAPSHOT_DIR / f'layer{TARGET_LAYER}.sae.pt'
    require_path(sae_file, f'SAE checkpoint for layer {TARGET_LAYER}')
    layer_device = next(model.model.layers[TARGET_LAYER].parameters()).device
    sae_state = torch.load(sae_file, map_location='cpu')
    W_enc = sae_state['W_enc'].to(device=layer_device, dtype=DTYPE).contiguous()
    b_enc = sae_state['b_enc'].to(device=layer_device, dtype=DTYPE).contiguous()
    print('loaded SAE layer:', TARGET_LAYER, 'device:', layer_device)
    print('W_enc:', tuple(W_enc.shape), W_enc.dtype, W_enc.device)
    print('b_enc:', tuple(b_enc.shape), b_enc.dtype, b_enc.device)


load_sae_for_layer(TARGET_LAYER)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:05<00:00, 72.18it/s]


target layer device: cuda:0
model dtype: torch.bfloat16
W_enc: (65536, 4096) torch.bfloat16 cuda:0
b_enc: (65536,) torch.bfloat16 cuda:0


## 4. 特征提取函数

这里不保存巨大的 `(seq_len, 65536)` 稠密激活矩阵，只保留每个 token 的 TopK feature id 和 activation value。

In [ ]:
def input_device_for_model(model):
    return model.get_input_embeddings().weight.device


def sae_topk_from_residual(residual: torch.Tensor, top_k: int = SAE_TOP_K):
    residual = residual.to(device=layer_device, dtype=DTYPE)
    pre_acts = residual @ W_enc.T + b_enc
    values, indices = torch.topk(pre_acts, k=top_k, dim=-1)
    return indices, values


def extract_one_input_ids(input_ids: list[int], return_tokens: bool = True) -> dict:
    captured = {}

    def hook_fn(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured['residual'] = hidden.detach()

    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=input_device_for_model(model))
    attention_mask = torch.ones_like(input_tensor)

    hook = model.model.layers[TARGET_LAYER].register_forward_hook(hook_fn)
    try:
        _ = model(input_ids=input_tensor, attention_mask=attention_mask, use_cache=False)
    finally:
        hook.remove()

    residual = captured['residual']
    feature_ids, feature_values = sae_topk_from_residual(residual)

    feature_ids = feature_ids[0].detach().cpu()
    feature_values = feature_values[0].detach().float().cpu()
    input_ids_cpu = input_tensor[0].detach().cpu()

    token_records = []
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu.tolist()) if return_tokens else [None] * len(input_ids_cpu)
    for pos, (tok, ids, vals) in enumerate(zip(tokens, feature_ids, feature_values)):
        token_records.append({
            'position': pos,
            'token': tok,
            'feature_ids': ids.tolist(),
            'feature_values': vals.tolist(),
        })

    return {
        'text': tokenizer.decode(input_ids_cpu.tolist(), skip_special_tokens=True),
        'num_tokens': len(input_ids_cpu),
        'token_records': token_records,
    }


def extract_one_text(text: str, return_tokens: bool = True) -> dict:
    captured = {}

    def hook_fn(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured['residual'] = hidden.detach()

    hook = model.model.layers[TARGET_LAYER].register_forward_hook(hook_fn)
    try:
        encoded = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        encoded = {k: v.to(input_device_for_model(model)) for k, v in encoded.items()}
        _ = model(**encoded, use_cache=False)
    finally:
        hook.remove()

    residual = captured['residual']
    feature_ids, feature_values = sae_topk_from_residual(residual)

    feature_ids = feature_ids[0].detach().cpu()
    feature_values = feature_values[0].detach().float().cpu()
    input_ids = encoded['input_ids'][0].detach().cpu()

    token_records = []
    tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist()) if return_tokens else [None] * len(input_ids)
    for pos, (tok, ids, vals) in enumerate(zip(tokens, feature_ids, feature_values)):
        token_records.append({
            'position': pos,
            'token': tok,
            'feature_ids': ids.tolist(),
            'feature_values': vals.tolist(),
        })

    return {
        'text': text,
        'num_tokens': len(input_ids),
        'token_records': token_records,
    }



## 5. 按 token 均衡，批量提取、保存、比较 feature 分布

三组数据先按完整 conversation tokenize，再抽样到相同 token 总量。后续的 `WINDOW_TOKEN_LENGTH` 只是模型前向计算窗口；feature 分布统计对所有 token 累加，不按 window/chunk 汇总。


In [ ]:
def tokenize_conversation_row(row: pd.Series) -> dict | None:
    token_ids = tokenizer(
        row[TEXT_COLUMN],
        add_special_tokens=False,
        truncation=False,
    )['input_ids']
    if not token_ids:
        return None
    return {
        **{col: row[col] for col in row.index if col != TEXT_COLUMN},
        'source_sample_id': row['sample_id'],
        'input_ids': token_ids,
        'num_tokens': len(token_ids),
        TEXT_COLUMN: row[TEXT_COLUMN],
    }


conversation_token_rows = []
for _, row in tqdm(conversation_df.iterrows(), total=len(conversation_df), desc='tokenize conversations'):
    tokenized = tokenize_conversation_row(row)
    if tokenized is not None:
        conversation_token_rows.append(tokenized)

conversation_token_df = pd.DataFrame(conversation_token_rows)
print('conversations before balancing:')
print(conversation_token_df.groupby('dataset').size())
print('tokens before balancing:')
print(conversation_token_df.groupby('dataset')['num_tokens'].sum())
print('avg tokens per conversation before balancing:')
print(conversation_token_df.groupby('dataset')['num_tokens'].mean())

if BALANCED_TOKENS_PER_DATASET is None:
    tokens_per_dataset = int(conversation_token_df.groupby('dataset')['num_tokens'].sum().min())
else:
    tokens_per_dataset = int(BALANCED_TOKENS_PER_DATASET)
print('target tokens per dataset:', tokens_per_dataset)


def make_processing_windows_for_seed(seed: int, label: str) -> pd.DataFrame:
    window_rows = []
    for dataset_name, part in conversation_token_df.groupby('dataset', sort=False):
        part = part.sample(frac=1, random_state=seed).reset_index(drop=True)
        remaining = tokens_per_dataset
        for _, conv in part.iterrows():
            if remaining <= 0:
                break
            selected_ids = conv['input_ids'][:remaining]
            if not selected_ids:
                continue
            remaining -= len(selected_ids)
            metadata = {col: conv[col] for col in conv.index if col not in {TEXT_COLUMN, 'input_ids', 'num_tokens'}}
            for window_idx, start in enumerate(range(0, len(selected_ids), WINDOW_TOKEN_LENGTH)):
                window_ids = selected_ids[start:start + WINDOW_TOKEN_LENGTH]
                if not window_ids:
                    continue
                window_rows.append({
                    **metadata,
                    'sample_id': f'{conv["sample_id"]}:window{window_idx}',
                    'window_id': window_idx,
                    'window_start_token': start,
                    'window_num_tokens': len(window_ids),
                    'input_ids': window_ids,
                    TEXT_COLUMN: tokenizer.decode(window_ids, skip_special_tokens=True),
                })

    balanced_df = pd.DataFrame(window_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    balanced_df['row_id'] = range(len(balanced_df))

    print(f'processing windows after balancing ({label}, seed={seed}):')
    print(balanced_df.groupby('dataset').size())
    print(f'tokens after balancing ({label}, seed={seed}):')
    print(balanced_df.groupby('dataset')['window_num_tokens'].sum())
    token_balance = balanced_df.groupby('dataset')['window_num_tokens'].sum()
    print(f'max/min token ratio after balancing ({label}):', token_balance.max() / token_balance.min())
    return balanced_df


for layer in TARGET_LAYERS:
    load_sae_for_layer(layer)
    layer_seed = RANDOM_SEED + TARGET_LAYER
    df = make_processing_windows_for_seed(layer_seed, label=f'layer{TARGET_LAYER}')
    OUTPUT_DIR = OUTPUT_BASE_DIR / f'ptsd_control_sae_layer{TARGET_LAYER}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    token_jsonl_path = OUTPUT_DIR / 'token_level_features.jsonl'
    ptsd_vs_empathetic_top_path = OUTPUT_DIR / f'ptsd_vs_empathetic_top{TOP_N}.csv'
    ptsd_vs_daily_top_path = OUTPUT_DIR / f'ptsd_vs_daily_top{TOP_N}.csv'

    group_activation_value_sums = defaultdict(float)
    group_activation_value_maxes = defaultdict(float)
    group_activation_counts = Counter()
    group_total_tokens = Counter()
    feature_content_token_counts = defaultdict(Counter)
    feature_noise_counts = Counter()
    feature_observed_counts = Counter()
    metadata_columns = [col for col in df.columns if col not in {TEXT_COLUMN, 'input_ids'}]

    STOPWORDS = {
        'a', 'an', 'and', 'are', 'as', 'at', 'be', 'but', 'by', 'for', 'from', 'had', 'has',
        'have', 'he', 'her', 'him', 'his', 'i', 'in', 'is', 'it', 'its', 'me', 'my', 'of',
        'on', 'or', 'our', 'she', 'so', 'that', 'the', 'their', 'them', 'they', 'this', 'to',
        'was', 'we', 'were', 'with', 'you', 'your',
        'am', 'do', 'does', 'did', 'done', 'can', 'could', 'would', 'should', 'will',
        'just', 'really', 'very', 'get', 'got', 'go', 'going', 'know', 'think', 'want',
        'one', 'all', 'also', 'if', 'then', 'there', 'here', 'what', 'when', 'where',
        'why', 'who', 'how', 'yes', 'no', 'not', 'nope', 'yeah', 'oh', 'ok', 'okay'
    }
    CONTRACTION_PARTS = {
        "'s", "'m", "'re", "'ve", "'d", "'ll", "'t", "n't",
        "s", "m", "re", "ve", "d", "ll", "t", "nt",
        "don", "doesn", "didn", "isn", "aren", "wasn", "weren",
        "couldn", "wouldn", "shouldn", "won", "cant", "cannot",
        "hasn", "haven", "hadn",
    }
    MOJIBAKE_MARKERS = {'芒模募', '臓', '鈻', 'âģļ', 'âĢĻ', 'âģ', 'âĢ'}
    PUNCT_CHARS = set('.,;:!?-()[]{}"\'*/\\|_+=<>@#$%^&~')


    def clean_metadata_value(value):
        if value is None:
            return None
        if isinstance(value, float) and pd.isna(value):
            return None
        if hasattr(value, 'item'):
            return value.item()
        return value


    def normalize_token_for_summary(token) -> str:
        token = '' if token is None else str(token)
        token = token.replace('Ġ', ' ').replace('▁', ' ')
        token = token.replace('<0x0A>', ' ').replace('Ċ', ' ')
        for marker in MOJIBAKE_MARKERS:
            token = token.replace(marker, ' ')
        return token.strip().lower()


    def is_noise_token(token: str) -> bool:
        token = normalize_token_for_summary(token)
        if not token:
            return True
        if token in STOPWORDS:
            return True
        if token in CONTRACTION_PARTS:
            return True
        if 'â' in token:
            return True
        if any('\u4e00' <= ch <= '\u9fff' for ch in token):
            return True
        if token.isdigit():
            return True
        if len(token) == 1 and not token.isalpha():
            return True
        return all(ch in PUNCT_CHARS for ch in token)


    def update_feature_token_summary(dataset_name: str, token_records: list[dict]):
        for rec in token_records:
            token = normalize_token_for_summary(rec['token'])
            noise = is_noise_token(token)
            for fid in rec['feature_ids']:
                fid = int(fid)
                key = (dataset_name, fid)
                feature_observed_counts[key] += 1
                if noise:
                    feature_noise_counts[key] += 1
                if token and not noise:
                    feature_content_token_counts[key][token] += 1


    with token_jsonl_path.open('w', encoding='utf-8') as f:
        for row_idx, row in tqdm(df.iterrows(), total=len(df), desc='extract SAE features'):
            dataset_name = row['dataset']
            row_metadata = {col: clean_metadata_value(row[col]) for col in metadata_columns}
            result = extract_one_input_ids(row['input_ids'])
            result.update(row_metadata)
            result['row_id'] = int(row_idx)
            f.write(json.dumps(result, ensure_ascii=False) + chr(10))

            update_feature_token_summary(dataset_name, result['token_records'])
            group_total_tokens[dataset_name] += int(result['num_tokens'])

            for rec in result['token_records']:
                for fid, value in zip(rec['feature_ids'], rec['feature_values']):
                    fid = int(fid)
                    activation_value = float(value)
                    key = (dataset_name, fid)
                    group_activation_value_sums[key] += activation_value
                    group_activation_value_maxes[key] = max(group_activation_value_maxes[key], activation_value)
                    group_activation_counts[key] += 1

    distribution_rows = []
    for (dataset_name, fid), activation_value_sum in group_activation_value_sums.items():
        n_tokens = group_total_tokens[dataset_name]
        activation_count = group_activation_counts[(dataset_name, fid)]
        distribution_rows.append({
            'dataset': dataset_name,
            'feature_id': fid,
            'dataset_token_count': n_tokens,
            'activation_count': activation_count,
            'activation_frequency_per_1k_tokens': activation_count / max(n_tokens, 1) * 1000,
            'activation_value_sum': activation_value_sum,
            'activation_value_max': group_activation_value_maxes[(dataset_name, fid)],
            'mean_activation_value_when_active': activation_value_sum / max(activation_count, 1),
        })

    dataset_distribution_df = pd.DataFrame(distribution_rows).sort_values(
        ['dataset', 'activation_frequency_per_1k_tokens', 'activation_value_sum'],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    token_summary_df = pd.DataFrame([
        {
            'dataset': dataset_name,
            'feature_id': fid,
            'observed_token_hits': feature_observed_counts[(dataset_name, fid)],
            'noise_token_hits': feature_noise_counts[(dataset_name, fid)],
            'noise_ratio': feature_noise_counts[(dataset_name, fid)] / max(feature_observed_counts[(dataset_name, fid)], 1),
            'top_content_tokens': '; '.join(f'{tok}:{count}' for tok, count in feature_content_token_counts[(dataset_name, fid)].most_common(20)),
        }
        for dataset_name, fid in group_activation_value_sums.keys()
    ]).sort_values(['dataset', 'noise_ratio', 'observed_token_hits'], ascending=[True, True, False]).reset_index(drop=True)


    def compare_feature_distribution(distribution_df: pd.DataFrame, token_summary_df: pd.DataFrame, control_dataset: str) -> pd.DataFrame:
        ptsd = distribution_df[distribution_df['dataset'] == 'ptsd'].set_index('feature_id')
        control = distribution_df[distribution_df['dataset'] == control_dataset].set_index('feature_id')
        all_features = sorted(set(ptsd.index) | set(control.index))

        rows = []
        for fid in all_features:
            p = ptsd.loc[fid] if fid in ptsd.index else None
            c = control.loc[fid] if fid in control.index else None
            ptsd_freq = float(p['activation_frequency_per_1k_tokens']) if p is not None else 0.0
            control_freq = float(c['activation_frequency_per_1k_tokens']) if c is not None else 0.0
            ptsd_count = int(p['activation_count']) if p is not None else 0
            control_count = int(c['activation_count']) if c is not None else 0
            rows.append({
                'feature_id': fid,
                'ptsd_activation_count': ptsd_count,
                f'{control_dataset}_activation_count': control_count,
                'ptsd_activation_frequency_per_1k_tokens': ptsd_freq,
                f'{control_dataset}_activation_frequency_per_1k_tokens': control_freq,
                'activation_frequency_diff_ptsd_minus_control': ptsd_freq - control_freq,
            })

        out = pd.DataFrame(rows)
        ptsd_tokens = token_summary_df[token_summary_df['dataset'] == 'ptsd'][['feature_id', 'noise_ratio', 'top_content_tokens']]
        ptsd_tokens = ptsd_tokens.rename(columns={'noise_ratio': 'ptsd_noise_ratio', 'top_content_tokens': 'ptsd_top_content_tokens'})
        control_tokens = token_summary_df[token_summary_df['dataset'] == control_dataset][['feature_id', 'noise_ratio', 'top_content_tokens']]
        control_tokens = control_tokens.rename(columns={'noise_ratio': f'{control_dataset}_noise_ratio', 'top_content_tokens': f'{control_dataset}_top_content_tokens'})
        out = out.merge(ptsd_tokens, on='feature_id', how='left').merge(control_tokens, on='feature_id', how='left')
        return out.sort_values(['activation_frequency_diff_ptsd_minus_control', 'ptsd_activation_frequency_per_1k_tokens'], ascending=False).reset_index(drop=True)


    ptsd_vs_empathetic_df = compare_feature_distribution(dataset_distribution_df, token_summary_df, 'empathetic')
    ptsd_vs_daily_df = compare_feature_distribution(dataset_distribution_df, token_summary_df, 'daily')

    def top_n_ptsd_specific(compare_df: pd.DataFrame, control_dataset: str, n: int = TOP_N) -> pd.DataFrame:
        diff_col = 'activation_frequency_diff_ptsd_minus_control'
        top = compare_df.sort_values(diff_col, ascending=False).head(n).copy()
        top['direction'] = 'ptsd_specific'
        top.insert(0, 'comparison', f'ptsd_vs_{control_dataset}')
        return top.reset_index(drop=True)


    ptsd_vs_empathetic_top_df = top_n_ptsd_specific(ptsd_vs_empathetic_df, 'empathetic')
    ptsd_vs_daily_top_df = top_n_ptsd_specific(ptsd_vs_daily_df, 'daily')
    ptsd_vs_empathetic_top_df.to_csv(ptsd_vs_empathetic_top_path, index=False)
    ptsd_vs_daily_top_df.to_csv(ptsd_vs_daily_top_path, index=False)

    feature_diff_top_df = pd.concat(
        [ptsd_vs_empathetic_top_df, ptsd_vs_daily_top_df],
        ignore_index=True,
    )

    print('saved:', ptsd_vs_empathetic_top_path)
    print('saved:', ptsd_vs_daily_top_path)
    print('rows per comparison:', len(ptsd_vs_empathetic_top_df), len(ptsd_vs_daily_top_df), 'top n:', TOP_N)
    display(ptsd_vs_empathetic_top_df.head(20))
    display(ptsd_vs_daily_top_df.head(20))


## 6. 查看某个 feature 在三组数据里的高激活 token


In [ ]:
ptsd_specific_features_to_inspect = feature_diff_top_df[feature_diff_top_df['direction'] == 'ptsd_specific']
FEATURE_ID_TO_INSPECT = int(ptsd_specific_features_to_inspect.iloc[0]['feature_id']) if len(ptsd_specific_features_to_inspect) else 0

hits = []
with token_jsonl_path.open('r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        if item.get('dataset') != 'ptsd':
            continue
        for rec in item['token_records']:
            if FEATURE_ID_TO_INSPECT in rec['feature_ids']:
                j = rec['feature_ids'].index(FEATURE_ID_TO_INSPECT)
                hits.append({
                    'dataset': item.get('dataset'),
                    'row_id': item['row_id'],
                    'sample_id': item.get('sample_id'),
                    'conversation_id': item.get('conversation_id'),
                    'phase': item.get('phase'),
                    'source_split': item.get('source_split'),
                    'source_sample_id': item.get('source_sample_id'),
                    'window_id': item.get('window_id'),
                    'window_start_token': item.get('window_start_token'),
                    'position': rec['position'],
                    'token': rec['token'],
                    'value': rec['feature_values'][j],
                    'text_preview': item['text'][:200],
                })

hits_df = pd.DataFrame(hits).sort_values('value', ascending=False).reset_index(drop=True)
print('PTSD feature_id:', FEATURE_ID_TO_INSPECT, 'PTSD hits:', len(hits_df))
display(feature_diff_top_df[feature_diff_top_df['feature_id'] == FEATURE_ID_TO_INSPECT][[
    'direction',
    'feature_id',
    'comparison',
    'activation_frequency_diff_ptsd_minus_control',
    'ptsd_activation_frequency_per_1k_tokens',
    'ptsd_top_content_tokens',
]])

display(hits_df.head(50)[['token', 'value', 'phase', 'conversation_id', 'window_id', 'position', 'text_preview']])

